# ANCOVA Analýza (s českými komentáři)
Tento notebook načte data z `ANCOVA.csv`, provede explorativní analýzu, ověří předpoklady ANCOVA, spustí model a zobrazí výsledky.


In [ ]:
# Import potřebných knihoven
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats


In [ ]:
# Načtení dat ze souboru ANCOVA.csv
df = pd.read_csv('ANCOVA.csv')
# Zobrazení prvních 5 řádků
df.head()

In [ ]:
# Popisné statistiky podle vyučovací metody
summary_stats = df.groupby('Teaching Method').agg({
    'Posttest Score': ['mean', 'std', 'min', 'max'],
    'Pretest Score': ['mean', 'std', 'min', 'max']
})
summary_stats

In [ ]:
# Vytvoření scatter plotu pro vztah Pretest vs Posttest podle metody
fig = px.scatter(df, x='Pretest Score', y='Posttest Score', color='Teaching Method', trendline='ols',
                 title='Pretest vs Posttest podle metody')
fig.show()

In [ ]:
# Ověření předpokladů ANCOVA
# 1. Homogenita regresních koeficientů (interakce mezi Pretest a metodou)
interaction_model = smf.ols('Posttest_Score ~ Pretest_Score * Teaching_Method',
                            data=df.rename(columns=lambda x: x.replace(' ', '_'))).fit()
interaction_model.summary()

# 2. Normalita reziduí pomocí Shapiro-Wilk testu
ancova_model = smf.ols('Posttest_Score ~ Pretest_Score + Teaching_Method',
                       data=df.rename(columns=lambda x: x.replace(' ', '_'))).fit()
shapiro_test = stats.shapiro(ancova_model.resid)
print('Shapiro-Wilk test:', shapiro_test)

In [ ]:
# Výsledky ANCOVA (ANOVA tabulka)
anova_table = sm.stats.anova_lm(ancova_model, typ=2)
anova_table

In [ ]:
# Výpočet adjustovaných průměrů při průměrném Pretest skóre
mean_pretest = df['Pretest Score'].mean()
pred_data = pd.DataFrame({
    'Pretest_Score': [mean_pretest, mean_pretest],
    'Teaching_Method': ['A', 'B']
})
pred_means = ancova_model.predict(pred_data)
print('Adjustované průměry pro metody A a B:', pred_means)